In [ ]:
# Import the string module (not used in this code, but available for string operations)
import string

# Import emoji library to remove emojis from text
import emoji

# Import TextBlob for spelling correction
from textblob import TextBlob

# Import contractions library to expand words like "can't" → "cannot"
import contractions

# Import regular expressions for text cleaning
import re

# Import spaCy for NLP preprocessing such as tokenization and lemmatization
import spacy

# Import LangChain text splitter for chunking long text
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import FAISS vector database
from langchain_community.vectorstores import FAISS

# Import HuggingFace embedding model
from langchain_huggingface import HuggingFaceEmbeddings

# Import Google's Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI


# Function to normalize and clean text
def normalize_text(text: str):

    # Convert all characters to lowercase
    text = text.lower()

    # Expand contractions such as "don't" → "do not"
    text = contractions.fix(text)

    # Replace multiple spaces with a single space
    text = re.sub(r'\s{2,}', ' ', text)

    # Remove all emojis from the text
    text = emoji.replace_emoji(text, replace='')

    # Remove punctuation and special characters
    text = re.sub(r'[^0-9a-zA-Z\s]', ' ', text)

    # Correct spelling mistakes using TextBlob
    text = str(TextBlob(text).correct())

    # Return the cleaned text
    return text


# Main RAG function
def rag_architecture(text, query):

    # Load the English spaCy model
    nlp = spacy.load("en_core_web_sm")

    # Clean the input text
    text = normalize_text(text)

    # Tokenize the cleaned text
    tokens = nlp(text)

    # Lemmatize words and remove stopwords
    updated_tokens = [
        token.lemma_
        for token in tokens
        if not token.is_stop
    ]

    # Join the processed tokens back into a sentence
    text = ' '.join(updated_tokens).strip()

    # Split the text into chunks of 100 characters with 20-character overlap
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20
    )

    # Convert chunks into LangChain Document objects
    chunks = splitter.create_documents([text])

    # Load HuggingFace embedding model
    embedding_model = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2'
    )

    # Create a FAISS vector database from the chunks
    vectordb = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Retrieve the most relevant chunks
    def r_search(query, k=3):

        # Perform similarity search in FAISS
        r_chunks = vectordb.similarity_search(
            query,
            k=k
        )

        # Remove duplicate chunks using a set
        r_chunks = {
            chunk.page_content
            for chunk in r_chunks
        }

        # Combine retrieved chunks into one string
        r_text = '\n'.join(r_chunks)

        # Return retrieved text
        return r_text

    # Generate answer using Gemini
    def g_text(r_text, query):

        # Import os module to access environment variables
        import os

        # Prompt template
        prompt = f"""
Answer the following question from the retrieved context.

Context:
{r_text}

Question:
{query}

Provide the answer in a structured manner.
"""

        # Load Gemini model
        llm_model = ChatGoogleGenerativeAI(
            model="gemini-3.5-flash",
            api_key=os.environ["GEMINI_API_KEY"]
        )

        # Send prompt to Gemini and extract only the generated text
        response = llm_model.invoke(prompt).content

        # Return the response
        return response

    # Retrieve relevant chunks
    r_response = r_search(query)

    # Generate final answer
    g_response = g_text(r_response, query)

    # Return generated response
    return g_response


# -------------------------
# Example Usage
# -------------------------

# Input document
# text = """
# Artificial Intelligence is the simulation of human intelligence by machines.
# Machine learning is a subset of AI.
# Deep learning is a subset of machine learning.
# """

text = open('Text_Normalization_and_RAG_Pipeline_Notes.txt', encoding='utf-8').read()
# text = normalize_text(text)

# User query
user_prompt = "What is Text Normalization?"

# Remove special characters from query
user_prompt = re.sub(r'[^0-9a-zA-Z\s]', '', user_prompt)

# Run the RAG pipeline
response = rag_architecture(text, user_prompt)

# Print the generated response
# print(response)
print(response[0]['text'])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3427.18it/s]


Based on the provided context, **Text Normalization** is characterized by the following points:

* **Core Process:** It involves converting text to lowercase.
* **Role in RAG Pipeline:** It represents the beginning of the input text pipeline where raw text is collected from documents.
* **Application:** It is used as a query cleaning/normalization step in machine learning.


In [4]:
text = open('Text_Normalization_and_RAG_Pipeline_Notes.txt', encoding='utf-8').read()

In [5]:
text[:500]

'Text Normalization and RAG Pipeline Notes\n\n1. Input Text\n\nThe pipeline begins with raw text collected from a document, webpage,\nPDF, or user input.\n\nExample: “I’m learning Machine Learning! 😊 It’s amazing, isn’t it??”\n\n------------------------------------------------------------------------\n\n2. Text Normalization\n\na. Convert to Lowercase\n\nMethod: text.lower()\n\nPurpose: Converts all characters to lowercase so that “Machine” and\n“machine” are treated as the same word.\n\nExample: Before: I’m Learnin'